# Cadeia de numerário: previsão probabilística + MILP estocástico + simulação rolante

**Pesquisa Operacional Aplicada · UFABC · 2026.3** — Thiago Bibiano da Silva

Este notebook executa o projeto especificado na proposta (`docs/especificacao.md`) e gera os artefatos que o painel HTML consome:

1. **Dados** — NN5 (111 séries de saques diários de ATMs, 18/03/1996 a 17/05/1998), 20 séries sorteadas com semente fixa.
2. **Previsão** — Chronos-2 zero-shot, 21 quantis, 10 dias à frente, janela expansiva.
3. **Calibração** — cobertura por cauda, PIT, cópula gaussiana (dependência entre ATMs e dias), margem da P0.
4. **Cenários e estoques de segurança** — 1.000 trajetórias → 30 cenários preservando as duas caudas; EM/PP das agências e da central.
5. **Backtest rolante** — políticas P0 a P4 decidindo toda noite; só o 1º estágio é executado.
6. **Resultados, sensibilidade e conferências** — custos, serviço, VSS, EVPI, mudanças de decisão, critérios de aceite.
7. **Artefatos** — 10 tabelas do contrato de dados (Parquet) + `painel-numerario.html` com o JSON embutido + resumo em markdown.

Cada etapa grava o que é caro em disco (no Colab, no Google Drive) e é pulada se já existir: dá para rodar de novo ou retomar depois de uma queda de sessão. Rode as células em ordem.

| Bloco | Período | Uso |
|---|---|---|
| Histórico | 18/03/1996 – 16/11/1997 | só contexto da previsão |
| Calibração | 17/11/1997 – 08/03/1998 | cobertura, cópula, margem da P0 |
| Aquecimento | 09/03 – 22/03/1998 | simulação rodando, resultado descartado |
| Teste | 23/03 – 17/05/1998 | todos os indicadores (56 decisões por política) |

## 1. Ambiente

In [ ]:
# ============================================================
# 1. AMBIENTE — Drive, repositório e dependências (idempotente)
# ============================================================
import os, sys, subprocess, time, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

EM_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ThiagoBibiano/poa-supply-chain-atm-cash"
REPO_COMMIT = None        # fixe um commit (ex.: "abc1234") para reprodutibilidade
USAR_DRIVE = True         # grava os artefatos no Google Drive (sobrevivem ao fim da sessão)

if EM_COLAB:
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_DIR = "/content/poa-supply-chain-atm-cash"
    if not os.path.exists(REPO_DIR):
        url = REPO_URL
        try:  # repositório privado: guarde um token em Secrets do Colab com o nome GITHUB_TOKEN
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, REPO_DIR], check=True)
    elif REPO_COMMIT is None:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)
    if REPO_COMMIT:
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_COMMIT], check=True)
else:
    REPO_DIR = os.environ.get("POA_REPO_DIR", os.path.abspath(".."))

# dependências com versão fixada (o PuLP 4 exige Python >= 3.12; abaixo disso, a 3.3.2)
from importlib import metadata

def versao(pacote):
    try:
        return metadata.version(pacote)
    except metadata.PackageNotFoundError:
        return None

FIXADAS = {"pulp": "4.2.0" if sys.version_info >= (3, 12) else "3.3.2", "highspy": "1.15.1"}
if EM_COLAB:
    FIXADAS["chronos-forecasting"] = "2.3.2"
    faltam = [f"{p}=={v}" for p, v in FIXADAS.items() if versao(p) != v]   # no Colab, garante a versão
else:
    faltam = [f"{p}=={v}" for p, v in FIXADAS.items() if versao(p) is None]  # localmente, só o que falta
faltam += [p for p in ("holidays", "pyarrow") if versao(p) is None]
if faltam:
    print("instalando:", " ".join(faltam))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltam], check=True)

if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
import numerario
from numerario import pipeline as pl, artefatos as art, metricas as met
from numerario.config import Config

try:
    import torch
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sem GPU (CPU)"
except Exception:
    GPU = "torch indisponível"
print(f"numerario {numerario.__version__} · Python {sys.version.split()[0]} · {GPU} · repositório em {REPO_DIR}")

## 2. CONFIG

Todos os parâmetros num só lugar. Os custos, tempos do GO e a escala monetária são **ilustrativos** (o NN5 não os tem): troque pelos da operação quando houver. `MODO`:

- `"completo"` — Chronos-2, 30 cenários, 60 s por noite (≈ 1h05 no P3; o resto leva minutos);
- `"rapido"` — Chronos-2, 15 cenários, 30 s por noite (para iterar; o P3 leva cerca de metade);
- `"teste_local"` — dados sintéticos e SeasonalNaive (só para testar o encadeamento, sem internet nem GPU).

In [ ]:
# ============================================================
# 2. CONFIG — único bloco de parâmetros
# ============================================================
MODO = os.environ.get("POA_MODO", "completo")      # "completo" | "rapido" | "teste_local"

cfg = Config()
# --- experimento e políticas
cfg.experimento.nome = "A"                         # "A" (empírico) | "B" (recicladores + guichê, sintético)
cfg.experimento.politicas = ["P0", "P1", "P2", "P3", "P4"]   # opcional: "P3_sem_EM" (preço do estoque mínimo)
# --- dados e previsão
cfg.caminhos.dados_fonte = "zenodo"                # "zenodo" | "huggingface" | "arquivo" | "sintetico"
cfg.previsao.modelo = "chronos2"                   # "chronos2" | "seasonal_naive"
cfg.previsao.chronos_id = "amazon/chronos-2"
cfg.previsao.chronos_revisao = None                # fixe um commit do Hugging Face para reprodutibilidade
cfg.previsao.covariaveis_calendario = True         # feriados da Inglaterra e do País de Gales
# --- escala monetária, rede e custos (ilustrativos)
cfg.rede.saque_medio_atm_rs_mil = 50.0             # saque médio de um ATM típico, em R$ mil/dia
cfg.rede.Theta_min_dia = 90.0                      # limite diário de minutos do GO
cfg.custos.K_C, cfg.custos.K_E = 600.0, 1500.0     # R$ por visita convencional e especial
cfg.custos.c_GO = 3.5                              # R$ por minuto do GO (margem perdida)
cfg.custos.taxa_anual = 0.15                       # CDI → custo de oportunidade h
cfg.custos.nivel_servico_alvo = 0.99               # ρ* → penalidade p pela razão crítica
cfg.es.alpha = 0.95                                # nível de serviço das tesourarias (EM/PP)
# --- cenários e solver
cfg.cenarios.n_cenarios = 30
cfg.solver.tempo_limite_s = 60
cfg.solver.gap_relativo = 0.01

if MODO == "rapido":
    cfg.cenarios.n_cenarios, cfg.solver.tempo_limite_s = 15, 30
elif MODO == "teste_local":
    cfg.caminhos.dados_fonte, cfg.previsao.modelo = "sintetico", "seasonal_naive"
    cfg.cenarios.n_cenarios = int(os.environ.get("POA_CENARIOS", 10))
    cfg.solver.tempo_limite_s = float(os.environ.get("POA_TEMPO_LIMITE", 30))
    if os.environ.get("POA_TESTE_FIM"):
        cfg.datas.teste_fim = os.environ["POA_TESTE_FIM"]

# onde gravar (no Colab, no Drive; localmente, em saidas/ dentro do repositório)
RAIZ = os.environ.get("POA_RAIZ") or ("/content/drive/MyDrive/poa_numerario" if EM_COLAB and USAR_DRIVE
                                      else os.path.join(REPO_DIR, "saidas"))
RAIZ = os.path.join(RAIZ, f"exp{cfg.experimento.nome}_{MODO}")
os.makedirs(RAIZ, exist_ok=True)
print("Artefatos em:", RAIZ)
display(pd.DataFrame(cfg.resumo(), columns=["Parâmetro", "Valor"]).style.hide(axis="index"))

## 3. Dados (NN5)

Download do registro 4656110 do Zenodo (versão **com** faltantes, conferida pelo MD5), leitor `.tsf` próprio, sorteio das 20 séries com semente fixa (entre as de poucas lacunas na janela de calibração e teste), imputação causal dos dias indisponíveis (mediana do mesmo dia da semana nas 4 semanas anteriores; zero conta como indisponível) e conversão para R$ mil pela escala do CONFIG. A previsão vê os dias indisponíveis como NaN; a simulação usa a série imputada como demanda verdadeira.

In [ ]:
# ============================================================
# 3. DADOS — download, seleção, imputação e escala
# ============================================================
proj = pl.preparar(cfg, RAIZ)
fonte = proj.diagnosticos["fonte_dados"]
print({k: v for k, v in fonte.items() if k in ("fonte", "md5", "md5_ok", "aviso")})
print("Feriados (lista fixa × pacote holidays):", proj.diagnosticos["feriados_divergencias"] or "sem divergências")
print(f"Escala: 1 unidade NN5 = R$ {proj.dados.escala:,.3f} mil · indisponíveis no aquecimento+teste: {proj.dados.meta['frac_indisponivel']:.2%}")
display(proj.dados.mapa_series.T)

fig, ax = plt.subplots(figsize=(12, 3.6))
jan = proj.dados.saques.loc["1997-09-01":cfg.datas.teste_fim]
ax.plot(jan.index, jan.sum(axis=1), lw=1, color="#2a78d6", label="saque total das 20 séries (R$ mil)")
for ini, fim, cor, nome in [(cfg.datas.calibracao_inicio, cfg.datas.calibracao_fim, "#eda100", "calibração"),
                            (cfg.datas.aquecimento_inicio, pd.Timestamp(cfg.datas.teste_inicio) - pd.Timedelta(days=1), "#9e9e9e", "aquecimento"),
                            (cfg.datas.teste_inicio, cfg.datas.teste_fim, "#1baf7a", "teste")]:
    ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color=cor, alpha=0.12, label=nome)
ax.set_title("Demanda da região e divisão do tempo"); ax.legend(loc="upper left", ncol=4, fontsize=8); ax.grid(alpha=.3)
plt.show()

## 4. Rede piloto

Custódia do BC (BB) → tesouraria central → 4 tesourarias de agência (GO + 3 ATMs internos cada) e 8 ATMs externos supridos direto pela central. Capacidades, tetos e reservas derivam do saque médio histórico (dias de cobertura no CONFIG).

In [ ]:
# ============================================================
# 4. REDE — pontos, capacidades, tetos e tempos do GO
# ============================================================
pontos = proj.rede.tabela_pontos(cfg)
display(pontos.round(1))
print(f"Θ = {proj.rede.Theta:.0f} min/dia · θ_CF = {proj.rede.theta_cf:.0f} min · Z_max = R$ {proj.rede.Zmax:,.0f} mil · R_c = R$ {proj.rede.Rc:,.0f} mil")

## 5. Previsão

Chronos-2 zero-shot (`predict_quantiles`, 21 quantis nativos, 10 passos), uma chamada por origem com todas as séries, contexto expansivo com NaN nos dias indisponíveis e covariáveis de calendário. O SeasonalNaive quantílico roda junto, como referência ("valor do Chronos em custo"). Cache em Parquet por origem: se a sessão cair, retoma de onde parou.

In [ ]:
# ============================================================
# 5. PREVISÃO — Chronos-2 e SeasonalNaive (com cache)
# ============================================================
modelos = list(dict.fromkeys([cfg.previsao.modelo, "seasonal_naive"]))
t0 = time.time()
pl.prever(proj, modelos)
for m, info in proj.diagnosticos["previsao"].items():
    print(m, {k: v for k, v in info.items() if k != "arquivo"})
print(f"{time.time() - t0:.0f} s")

# exemplo: leque da primeira noite de teste para um ATM externo
conj = proj.previsoes[cfg.previsao.modelo]
k0, atm = proj.k_teste_ini - 1, proj.rede.externos[0]
q = conj.quantis(k0)[proj.idx.i(atm)]
dias = proj.cal.datas[k0 + 1:k0 + 1 + conj.H]
niv = np.array(conj.niveis)
fig, ax = plt.subplots(figsize=(9, 3.2))
for lo, hi, a in [(0.05, 0.95, .15), (0.25, 0.75, .3)]:
    ax.fill_between(dias, q[:, np.argmin(abs(niv - lo))], q[:, np.argmin(abs(niv - hi))], color="#2a78d6", alpha=a, lw=0)
ax.plot(dias, q[:, np.argmin(abs(niv - .5))], color="#2a78d6", label="P50")
ax.plot(dias, proj.real[atm].values[k0 + 1:k0 + 1 + conj.H], "o", color="#eb6834", label="real")
ax.set_title(f"{atm}: previsão feita em {proj.cal.data(k0).date()} (P05–P95, P25–P75, P50)"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## 6. Calibração (só a janela de calibração)

- **Cobertura por cauda**: Q01/Q05/Q10 e Q90/Q95/Q99 separadas — uma previsão pode estar boa no P99 e ruim no P05.
- **PIT** e **dependência**: z = Φ⁻¹(PIT), correlação entre ATMs e entre horizontes (Kronecker) com encolhimento; ρ médio alimenta a aproximação do estoque mínimo.
- **P0**: margem sobre a média histórica escolhida pelo menor custo simulado na calibração.

Nada da janela de teste entra aqui.

In [ ]:
# ============================================================
# 6. CALIBRAÇÃO — cópula, ρ, parâmetros da região e margem da P0
# ============================================================
cal_diag = pl.calibrar(proj)
display(pd.Series(cal_diag).to_frame("valor"))
print("Custo da P0 na calibração por margem:", {k: round(v) for k, v in proj.diagnosticos.get("calibracao_p0", {}).items()})

tabs_cob = []
for m in proj.previsoes:
    mp = met.metricas_previsao(proj.previsoes[m], proj.contexto, proj.origens_calibracao, proj.k_teste_ini - 1,
                               feriados=proj.cal.feriado)
    c = mp["caudas"].assign(modelo=m)
    tabs_cob.append(c)
    print(f"{m}: WQL {mp['WQL']:.3f} · MASE {mp['MASE']:.3f} · cobertura 80% {mp['cobertura_80']:.1%} · 90% {mp['cobertura_90']:.1%}")
display(pd.concat(tabs_cob).pivot_table(index=["cauda", "nominal"], columns="modelo", values="empirica").style.format("{:.1%}"))

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
Zc = pl.pits(proj, cfg.previsao.modelo, proj.origens_calibracao, proj.k_teste_ini - 1)
from scipy import stats
u = stats.norm.cdf(Zc[~np.isnan(Zc)])
axs[0].hist(u, bins=10, range=(0, 1), color="#2a78d6", alpha=.8, density=True); axs[0].axhline(1, color="k", ls="--", lw=1)
axs[0].set_title("PIT na calibração (plano = calibrado)")
im = axs[1].imshow(proj.dep.C_series, vmin=-1, vmax=1, cmap="RdBu_r")
axs[1].set_title(f"Correlação entre ATMs (cópula) · ρ médio = {proj.dep.rho_medio:.2f}")
axs[1].set_xticks(range(len(proj.idx.series))); axs[1].set_xticklabels(proj.idx.series, rotation=90, fontsize=7)
axs[1].set_yticks(range(len(proj.idx.series))); axs[1].set_yticklabels(proj.idx.series, fontsize=7)
plt.colorbar(im, ax=axs[1], fraction=.04); plt.tight_layout(); plt.show()

## 7. Cenários e estoques de segurança

Para cada noite: 1.000 trajetórias da cópula → 30 cenários (k-medoides) com as duas caudas preservadas (os 5% de demanda mais baixa e mais alta no período de proteção entram como grupos próprios). Das trajetórias saem o estoque mínimo e o ponto de pedido de cada tesouraria (quantil **da soma**), o estoque concentrado da central e o ganho de centralização.

In [ ]:
# ============================================================
# 7. CENÁRIOS E ESTOQUES DE SEGURANÇA — por noite (com cache)
# ============================================================
t0 = time.time()
pl.montar_noites(proj)
print(f"{len(proj.noites)} noites · {time.time() - t0:.0f} s")
em_tab = pl.tabela_estoque_minimo(proj)
teste = em_tab[em_tab["data"] >= pd.Timestamp(cfg.datas.teste_inicio)]
display(teste.groupby("ponto")[["dias_protecao", "saida_mediana", "em_formula", "em_independencia", "em_soma_quantis", "pp"]].mean().round(1))
cd = teste[teste["ponto"] == "CD"]
print(f"Ganho de centralização médio no teste: R$ {cd['ganho_centralizacao'].mean():,.0f} mil "
      f"({(cd['ganho_centralizacao'] / cd['total_separado']).mean():.0%} do que cada ponto guardaria sozinho)")
print(f"Lote de Baumol Q* = R$ {proj.calc_em.Q_baumol:,.0f} mil · Miller–Orr z − L = R$ {proj.calc_em.mo_z_menos_L:,.0f} mil")

# cenários de uma noite: trajetórias, cenários escolhidos e as duas caudas
cen = proj.noites[proj.k_teste_ini - 1]
reg = [i for i, s in enumerate(proj.idx.series) if ":" not in s]
tot_traj = cen.trajetorias[:, reg, :].sum(axis=1); tot_cen = cen.cenarios[:, reg, :].sum(axis=1)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].plot(tot_traj[:200].T, color="#9e9e9e", alpha=.15, lw=.7)
cores = {"cauda_baixa": "#2a78d6", "miolo": "#1baf7a", "cauda_alta": "#e34948"}
for s in range(len(cen.probs)):
    axs[0].plot(tot_cen[s], color=cores[cen.info["grupos"][s]], lw=1 + 30 * cen.probs[s], alpha=.9)
axs[0].set_title("Demanda da região: trajetórias (cinza) e cenários (cor = grupo, espessura = prob.)")
for b in proj.rede.tesourarias:
    g = teste[teste["ponto"] == b]
    axs[1].plot(g["data"], g["em_formula"], label=f"EM {b}")
axs[1].plot(cd["data"], cd["em_formula"], "k", lw=2, label="EM central")
axs[1].set_title("Estoque mínimo por noite (R$ mil)"); axs[1].legend(fontsize=7, ncol=3); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Backtest rolante (P0–P4)

Toda noite, de 08/03 a 16/05/1998: a política lê o estado (saldos e pedidos em trânsito), decide, e só o 1º estágio é executado contra a demanda real do dia seguinte. P1, P3 e P4 resolvem o MILP de dois estágios (R1–R14) com HiGHS; P0 e P2 são regras. O P3 parte da solução determinística da mediana e mantém binárias só nos 2 primeiros dias (o resto é relaxado); com 30 cenários, a maioria das noites para no limite de tempo com gap de alguns por cento (a saída abaixo e o painel mostram tempo e gap). Checkpoint a cada 7 noites: se a sessão cair, rode a célula de novo.

In [ ]:
# ============================================================
# 8. BACKTEST — uma política por vez, com checkpoint
# ============================================================
t0 = time.time()
resultados = pl.rodar(proj)
print(f"backtest: {time.time() - t0:.0f} s")
for n, r in resultados.items():
    ex = r.execucoes
    milp = ex[ex["status"] != "regra"]
    gap = milp["gap"].astype(float)
    extra = (f" · {len(milp[milp['status'] == 'Optimal'])}/{len(milp)} noites ótimas · {milp['segundos'].mean():.1f} s/noite"
             f" · gap médio {gap.mean():.1%} (máx. {gap.max():.1%}) · {int(ex['fallback'].sum())} noites na regra P0"
             if len(milp) else "")
    print(f"{n}: {len(r.estados['data'].unique())} dias simulados{extra}")

## 9. Resultados (janela de teste)

In [ ]:
# ============================================================
# 9. RESULTADOS — custo, serviço, capital, VSS/EVPI e replanejamento
# ============================================================
K_INI, K_FIM = proj.k_teste_ini, proj.k_teste_fim
cons = art.consolidar(proj, resultados, K_INI, K_FIM)
kp, comp = cons["kpis"], cons["comparacao"]
tab = pd.DataFrame({n: {
    "custo total (R$)": k["total"], "custo do erro (R$)": comp["por_politica"].get(n, {}).get("custo_erro"),
    "nível de serviço": k["fill"], "rupturas (ponto-dia)": k["rupt"], "tesouraria abaixo do EM": k["abaixoMin"],
    "saldo parado (ponto-dia)": k["parado"], "especiais": k["esp"], "convencionais": k["conv"],
    "minutos GO/agência-dia": k["goAvg"], "da custódia (R$ mil)": k["bcTot"], "saldo médio da rede (R$ mil)": k["saldo_medio_rede"],
    "mudanças de decisão": k["chg"], "nervosismo do plano": k["nervosismo"]["taxa"], "especiais reativas": k["especiais_reativas"],
    "divergência do oráculo": k.get("divergencia_oraculo")} for n, k in kp.items()})
display(tab.style.format(precision=3, thousands=".", decimal=","))
display(Markdown(f"**VSS** = custo(P1) − custo(P3) = R$ {comp['VSS']:,.0f} · **EVPI** = custo(P3) − custo(P4) = R$ {comp['EVPI']:,.0f}"
                 if comp["VSS"] is not None and comp["EVPI"] is not None else "VSS/EVPI: rode P1, P3 e P4."))

cores = {"conv": "#2a78d6", "esp": "#eb6834", "go": "#1baf7a", "opor": "#eda100", "rupt": "#e34948", "estouro": "#e87ba4", "num": "#008300"}
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
base = np.zeros(len(kp))
for c, cor in cores.items():
    v = np.array([kp[n]["comp"][c] for n in kp])
    axs[0].bar(list(kp), v, bottom=base, color=cor, label=c); base += v
axs[0].set_title("Custo total por componente (R$)"); axs[0].legend(fontsize=8, ncol=4)
for n, r in resultados.items():
    cus = r.custos[(r.custos["k"] >= K_INI) & (r.custos["k"] <= K_FIM)].groupby("data")["valor"].sum().cumsum()
    axs[1].plot(cus.index, cus.values, label=n)
axs[1].set_title("Custo acumulado no teste (R$)"); axs[1].legend(); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

mp = cons["previsao"]
display(pd.DataFrame({k: {"WQL": v["WQL"], "MASE": v["MASE"], "viés": v["vies_pct"], "cobertura 80%": v["cobertura_80"],
                          "cobertura 90%": v["cobertura_90"]} for k, v in mp.items() if v.get("n")}).T.round(4))

## 10. Sensibilidade (opcional)

Uma variação por vez (seção 8.5), com previsões e dependência reaproveitadas e cenários reduzidos (10) para caber no tempo. **Custo × Θ** dá o valor de um minuto do GO por diferença finita — o dual de R5 com as binárias fixadas é zero por construção (R5 só tem binárias). **Custo × ρ*** traça a curva custo × nível de serviço.

In [ ]:
# ============================================================
# 10. SENSIBILIDADE — desligada por padrão (cada ponto roda P1 e P3)
# ============================================================
RODAR_SENSIBILIDADE = False
sens = None
if RODAR_SENSIBILIDADE:
    grade = {"rede.Theta_min_dia": [60.0, 90.0, 120.0],
             "custos.nivel_servico_alvo": [0.95, 0.99, 0.995]}
    sens = pl.sensibilidade(proj, grade, politicas=["P1", "P3"], n_cenarios=10)
    display(sens.round(4))
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
    for (par, pol), g in sens.groupby(["parametro", "politica"]):
        ax = axs[0] if par == "rede.Theta_min_dia" else axs[1]
        x = g["valor"] if par == "rede.Theta_min_dia" else g["nivel_servico"]
        ax.plot(x, g["custo_total"], "o-", label=pol)
    axs[0].set_title("Custo total × limite de minutos do GO (Θ)"); axs[1].set_title("Custo × nível de serviço")
    for ax in axs: ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 11. Conferências (critérios de aceite do painel)

In [ ]:
# ============================================================
# 11. CONFERÊNCIAS — os critérios de aceite, checados nos dados
# ============================================================
conf = art.conferencias(proj, resultados, cons, K_INI, K_FIM)
display(pd.DataFrame(conf).assign(ok=lambda d: d["ok"].map({True: "✓", False: "✗"})))

## 12. Artefatos e painel HTML

Grava em `RAIZ/05_artefatos/`:

- `*.parquet` — as 10 tabelas do contrato de dados (pontos, estoque_minimo, custodia, previsoes, realizado, decisoes, estados, custos, modelo, execucoes), para análises futuras;
- `painel.json` — o JSON que o painel consome (dá para regenerar o HTML sem rodar nada);
- `painel-numerario.html` — página única, JavaScript puro e SVG, com o JSON embutido;
- `resumo_para_slides.md` — esqueleto em markdown com os números principais;
- `manifesto.json` — configuração completa e hashes dos arquivos (origem dos dados e diagnósticos em `RAIZ/06_execucoes/diagnosticos.json`).

In [ ]:
# ============================================================
# 12. ARTEFATOS — tabelas, JSON, painel HTML e resumo
# ============================================================
import hashlib
PASTA = os.path.join(RAIZ, "05_artefatos")
tabelas = art.tabelas_contrato(proj, resultados, K_INI, K_FIM)
arqs = art.gravar_tabelas(tabelas, PASTA)
dados_painel = art.json_painel(proj, resultados, cons, conf, K_INI, K_FIM, sensibilidade=sens)
art.gravar_json(dados_painel, os.path.join(PASTA, "painel.json"))     # JSON estrito (NaN → null)
html = art.escrever_painel(dados_painel, os.path.join(REPO_DIR, "painel", "template.html"),
                           os.path.join(PASTA, "painel-numerario.html"))
with open(os.path.join(PASTA, "resumo_para_slides.md"), "w", encoding="utf-8") as fh:
    fh.write(art.resumo_markdown(proj, cons, conf))
pl.salvar_diagnosticos(proj)
manifesto = {"gerado_em": time.strftime("%Y-%m-%d %H:%M"), "modo": MODO, "config": cfg.to_dict(),
             "arquivos": {os.path.basename(a): hashlib.sha256(open(a, "rb").read()).hexdigest()[:16]
                          for a in list(arqs.values()) + [html]}}
art.gravar_json(manifesto, os.path.join(PASTA, "manifesto.json"), indent=1)
print("Painel:", html, f"({os.path.getsize(html) / 1e6:.1f} MB)")
display(pd.Series({k: v.shape for k, v in tabelas.items()}, name="linhas × colunas").to_frame())
if EM_COLAB:
    from google.colab import files
    files.download(html)    # baixa o painel; ele também fica no Drive

### Como consumir os artefatos depois

```python
import pandas as pd
estados = pd.read_parquet(f"{RAIZ}/05_artefatos/estados.parquet")      # saldos e fluxos por política, ponto e dia
custos  = pd.read_parquet(f"{RAIZ}/05_artefatos/custos.parquet")       # custo por componente
prev    = pd.read_parquet(f"{RAIZ}/05_artefatos/previsoes.parquet")    # 21 quantis por ponto, origem e alvo

# regenerar o HTML a partir do JSON salvo (sem rodar nada):
import json; from numerario import artefatos as art
art.escrever_painel(json.load(open(f"{RAIZ}/05_artefatos/painel.json")),
                    f"{REPO_DIR}/painel/template.html", f"{RAIZ}/05_artefatos/painel-novo.html")
```

In [ ]:
# visualização do painel dentro do notebook (opcional)
MOSTRAR_PAINEL = False
if MOSTRAR_PAINEL:
    from IPython.display import IFrame
    display(HTML(open(html, encoding="utf-8").read()))